# Survival

> I'm barely alive

In [ ]:
#| default_exp survival

In [ ]:
#| export
import torch, torch.nn as nn, lightning.pytorch as pl, warnings, math, numpy as np

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from wavefm.layers import Patch
from wavefm.loss import demographic_adversarial_loss, Momentum, nll_logistic_hazard
from wavefm.utils import trunc_normal_
#from torchsurv.loss.momentum import Momentum
from torchsurv.loss.weibull import log_hazard
from torchsurv.metrics.auc import Auc
from torchsurv.metrics.cindex import ConcordanceIndex
from torchsurv.stats.ipcw import get_ipcw

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.util import Surv

In [ ]:
#| export
def discretize_time(time_days, bin_years=0.25, min_year=0.0, max_year=10.0):
    """
    Discretize a tensor of observed times (in days) into bin indices for discrete-time survival.

    Args:
        time_days (Tensor): observed times in days (event or censoring).
        bin_years (float): width of bins in years (e.g., 0.25 = quarter-year).
        min_year (float): lower bound of the window in years.
        max_year (float): upper bound of the window in years.

    Returns:
        bin_idx (Tensor): indices in [0, n_bins-1] for each input time
    """
    days_per_bin = bin_years * 365.25
    min_days = min_year * 365.25
    max_days = max_year * 365.25

    time_days = torch.clamp(time_days, min=min_days, max=max_days)
    
    idx = torch.ceil(time_days / days_per_bin) - 1
    idx = idx.long()

    n_bins = int(np.ceil((max_days - min_days) / days_per_bin))
    idx = torch.clamp(idx, min=0, max=n_bins - 1)
    return idx

def make_cuts_days(min_year: float,
                   max_year: float,
                   bin_years: float):
    """
    Build right-edges of bins (in DAYS) for the discrete-time hazard model.

    Returns:
        cuts_days (np.ndarray): shape [J], right edges in days.
    """
    edges_years = np.arange(min_year + bin_years,
                            max_year + 1e-9,
                            bin_years)
    return edges_years * 365.25

def discretize_time_convert_logits_to_estimates(preds, new_time_days, cuts_days):
        preds = torch.sigmoid(preds)
        preds = torch.cumprod(1.0-preds,dim=1) # survival
        # map each new_time to its bin index (right-closed)
        idx = np.searchsorted(cuts_days, new_time_days.cpu().numpy(), side="right") - 1
        idx = np.clip(idx, 0, len(cuts_days) - 1).astype(int)
        idx_t = torch.as_tensor(idx, device=preds.device, dtype=torch.long)  # [n_times]

        S_at = preds.index_select(1, idx_t)            # [B, n_times]
        estimate = 1.0 - S_at                      # risk by each new_time
        return estimate

def hazards_to_survival(preds):
    preds = np.exp(np.log(1 - preds + 1e-8).cumsum(axis=1))
    return preds

def hazards_to_survival_torch(preds):
    preds = torch.exp(torch.log(1 - preds + 1e-8).cumsum(dim=1))
    return preds

In [ ]:
#| notest
# Yearly bins from 0–10y
cuts_days = make_cuts_days(0, 10, 1)
# edges_years = [1,2,3,...,10]
# bins = (0,1], (1,2], ..., (9,10]

# Times
t = torch.tensor([100., 365., 3700.])  # ~0.27y, 1y, ~10.1y
idx = discretize_time(t, bin_years=1, min_year=0, max_year=10)
# -> [0, 0, 9]  (last one clipped at 10y)
print(cuts_days), print(idx)

[ 365.25  730.5  1095.75 1461.   1826.25 2191.5  2556.75 2922.   3287.25
 3652.5 ]
tensor([0, 0, 9])


(None, None)

## Cox w Momentum

In [ ]:
#| export
class PatchTFTSurvivalDemo(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of gpus to use
                linear_probing_head, # model head to linear probe/train
                preloaded_model, # loaded pretrained model to use for linear probing
                evaluate_risk_years = [1,5,10],
                discretize_time=True, # indicator to discretize time for discrete-time survival
                discrete_time_bins=0.25, # function to discretize time for discrete-time survival
                time_range_years=(0,15), # events after this time are censored
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0.001, # weight decay for Adam optimizer
                final_weight_decay=1e-6, # weight decay for final layer
                use_weight_decay_scheduler=False,
                demographic_embeddings={'age_bin_idx': 9,'bmi_bin_idx': 7,'gender': 2},
                age_mlp_hidden_size=32,
                demographics_only=False,
                scheduler_kwargs={},
                transforms=None,
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.discrete_time_bins = discrete_time_bins
        self.time_range_years = time_range_years
        self.discretize_time = discretize_time
        self.demographic_embeddings = demographic_embeddings
        self.demographics_only = demographics_only
        self.transforms = transforms
        if evaluate_risk_years is not None:
            warnings.warn("evaluate_risk_years is ignored when discretize_time=True. Risk will be evaluated at all bins.")
        self.new_time = torch.tensor(make_cuts_days(self.time_range_years[0], self.time_range_years[1], self.discrete_time_bins), dtype=torch.float32)
        self.evaluate_risk_years = self.new_time / 365.25
        
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        if not self.demographics_only:
            self.feedforward = linear_probing_head
        else:
            self.feedforward = nn.Identity()

        self.time_bias = nn.Parameter(torch.zeros(len(self.new_time)))
        
        if "age_bin_idx" in demographic_embeddings:
            self.age_time_interaction = nn.Embedding(
                demographic_embeddings['age_bin_idx'],  # 9 age bins
                len(self.new_time)  # One effect per time bin
            )
            nn.init.zeros_(self.age_time_interaction.weight)

        elif "age_s1" in demographic_embeddings:
            self.age_time_interaction = nn.Sequential(nn.Linear(1, age_mlp_hidden_size), nn.ReLU(), nn.Dropout(0.1), nn.Linear(age_mlp_hidden_size, len(self.new_time)))
        
        else:
            self.age_time_interaction = nn.Identity()

        self.demographic_effects = nn.ParameterDict()
        for k, v in demographic_embeddings.items():
            if k not in ['age_bin_idx', 'age_s1']:
                self.demographic_effects[k] = nn.Parameter(torch.zeros(v))


        self.train_ipcw_events = []
        self.train_ipcw_times = []
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x, y, time, demographics):
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        time = discretize_time(time, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        if self.demographics_only:
            log_hz_k = self.time_bias
        else:
            log_hz_k = self.feedforward(x) + self.time_bias
        
        for i, k in enumerate(self.demographic_embeddings):
            if k == 'age_bin_idx':
                #print(demographics.shape, demographics[:, i].shape, demographics[:, i])
                age_emb = self.age_time_interaction(demographics[:, i].to(int)) # demographics[:, i] is [bs]
                log_hz_k = log_hz_k + age_emb
            elif k == 'age_s1':
                age_emb = self.age_time_interaction(demographics[:, i].unsqueeze(1)) # demographics[:, i] is [bs] -> unsqueeze to [bs, 1] for linear layer
                log_hz_k = log_hz_k + age_emb
            else:
                log_hz_k = log_hz_k + self.demographic_effects[k][demographics[:, i].to(int)].unsqueeze(1)
        return log_hz_k
          
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): 
        x, y, time, demographic_targets = batch
        log_hz_k = self(x, y, time, demographic_targets)
        return log_hz_k, y, time, demographic_targets

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, time, demographic_targets = batch
        self.train_ipcw_events.append(y.bool().detach().cpu())
        self.train_ipcw_times.append(time.detach().cpu())

        if self.transforms is not None:
            batch = self.transforms(batch)
            x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        
        if self.transforms is not None:
            time_pre_mix = self.train_ipcw_times[-1]
            time_pre_mix = torch.clamp(time_pre_mix, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
            time_bins_pre_mix = discretize_time(time_pre_mix, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
            # concat unmixed and mixed time bins for loss calculation
            time_bins = torch.stack([time_bins.squeeze(-1), time_bins_pre_mix.squeeze(-1).to(time_bins.device)], dim=1)

        loss = nll_logistic_hazard(log_hz_k, y, time_bins)
        
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time, demographic_targets = batch
        
        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        # use clamped time for loss calculation
        loss = nll_logistic_hazard(log_hz_k, y, time_bins) # [bs, n_classes, pred_len_seconds]

        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        self.events.append(y.bool().detach().cpu())
        self.times.append(time.detach().cpu()) # append original time
        self.preds.append(log_hz_k.detach())
        return loss
        # if torch.any(y):
        #     cindex = self.cindex(log_hz_k, y, time)
        #     self.log("cindex", cindex, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        # else:
        #     self.log("cindex", torch.tensor(0.0, device=x.device), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def on_validation_epoch_end(self):

        preds = torch.cat(self.preds).squeeze()
        preds = torch.sigmoid(preds)
        
        og_device = preds.device.type

        preds = preds.cpu().to(torch.float32).numpy()
        events = torch.cat(self.events).squeeze().numpy()
        times = torch.cat(self.times).squeeze().numpy()
        new_time = self.new_time.cpu().numpy()
        evaluate_risk_years = self.evaluate_risk_years.cpu().numpy()
        
        preds = 1 - hazards_to_survival(preds) # convert hazards to cumulative risk

        try:
            # perform IPCW censoring adjustment/ time clipping
            min_time, max_time = times.min(), times.max()
            if new_time.min() < min_time or new_time.max() > max_time:
                warnings.warn(f"Some evaluation times are outside the range of observed times. Clipping new_time from [{new_time.min()}, {new_time.max()}] to [{min_time}, {max_time}]")
                idx_ = np.where((new_time >= min_time) & (new_time <= max_time))[0]
                preds = preds[:, idx_] # subset preds to these indices
                new_time = new_time[idx_] # subset new_time to these indices
                evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices

            min_first_event_time = times[events == True].min() # get first positive event time (you cant evaluate before this) (but you can after)

            idx_ = np.where((new_time >= min_first_event_time))[0] # find indices of new_time after min_time
            preds = preds[:, idx_] # subset preds to these indices
            new_time = new_time[idx_] # subset new_time to these indices, so evaluations is only after min_time
            evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices
            
            tau = new_time[-1] #times[events == True].max() # time to evaluate c-index at (probability of censoring should be > 0 after this time)
            print(f"Events shape: {events.shape}, sum: {events.sum()}")
            print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
            print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")
        except Exception as e:
            print(f"IPCW or time clipping error: {str(e)}")


        try:
            # create structured arrays for sksurv
            train_events = torch.cat(self.train_ipcw_events).squeeze().numpy()
            train_times = torch.cat(self.train_ipcw_times).squeeze().numpy()

            # clip validation times to 
            
            train_structured = Surv.from_arrays(event=train_events, time=train_times)
            val_structured = Surv.from_arrays(event=events, time=times)
        except Exception as e:
            print(f"Structured array creation error: {str(e)}")
        try:
            # calculate metrics
            auc_all, mean_auc = cumulative_dynamic_auc(train_structured, val_structured, preds, new_time)
            auc_all = torch.tensor(auc_all, device=og_device)
            mean_auc = torch.tensor(mean_auc, device=og_device)
        except Exception as e:
            print(f"AUC calculation error: {str(e)}")
            auc_all = torch.tensor([0.0 for _ in evaluate_risk_years], device=og_device)
            mean_auc = torch.tensor(0.0, device=og_device)
        finally:
            self.log_dict({f"{y}year_auc_ipcw": score for y, score in zip(evaluate_risk_years, auc_all)}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log("mean_ipcw_auc", mean_auc, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        try:
            cindex = concordance_index_ipcw(train_structured, val_structured, preds[:,-1], tau=tau)[0]
            cindex = torch.tensor(cindex, device=og_device)
            self.log("cindex_ipcw", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index calculation error: {str(e)}")
        try:
                # calculate survival probabilities
            preds = 1 - preds # convert cumulative risk to survival
                # calculate brier score
            brier = integrated_brier_score(train_structured, val_structured, preds, new_time)
            brier = torch.tensor(brier, device=og_device)
            self.log("brier_score", brier, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"Brier calculation error: {str(e)}")
        try:
            all_score = mean_auc + (1 - brier) + cindex
            self.log("all_score", all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"All score calculation error: {str(e)}")
        self.preds.clear()
        self.events.clear()
        self.times.clear()

    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
             {
                'params': (self.time_bias)
            },
            {
                'params': (p for n, p in self.demographic_effects.named_parameters()) 
            },
            {
                'params': (p for n, p in self.age_time_interaction.named_parameters())
            }
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class SurvivalDemo(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of gpus to use
                evaluate_risk_years = [1,5,10],
                discretize_time=True, # indicator to discretize time for discrete-time survival
                discrete_time_bins=0.25, # function to discretize time for discrete-time survival
                time_range_years=(0,15), # events after this time are censored
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0.001, # weight decay for Adam optimizer
                final_weight_decay=1e-6, # weight decay for final layer
                use_weight_decay_scheduler=False,
                demographic_embeddings={'age_bin_idx': 9,'bmi_bin_idx': 7,'gender': 2},
                age_mlp_hidden_size=32,
                scheduler_kwargs={},
                transforms=None,
                ):
        super().__init__()
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.discrete_time_bins = discrete_time_bins
        self.time_range_years = time_range_years
        self.discretize_time = discretize_time
        self.demographic_embeddings = demographic_embeddings
        self.transforms = transforms
        if evaluate_risk_years is not None:
            warnings.warn("evaluate_risk_years is ignored when discretize_time=True. Risk will be evaluated at all bins.")
        self.new_time = torch.tensor(make_cuts_days(self.time_range_years[0], self.time_range_years[1], self.discrete_time_bins), dtype=torch.float32)
        self.evaluate_risk_years = self.new_time / 365.25

        self.time_bias = nn.Parameter(torch.zeros(len(self.new_time)))
        
        if "age_bin_idx" in demographic_embeddings:
            self.age_time_interaction = nn.Embedding(
                demographic_embeddings['age_bin_idx'],  # 9 age bins
                len(self.new_time)  # One effect per time bin
            )
            nn.init.zeros_(self.age_time_interaction.weight)

        elif "age_s1" in demographic_embeddings:
            self.age_time_interaction = nn.Sequential(nn.Linear(1, age_mlp_hidden_size), nn.ReLU(), nn.Dropout(0.1), nn.Linear(age_mlp_hidden_size, len(self.new_time)))
        
        else:
            self.age_time_interaction = nn.Identity()

        self.demographic_effects = nn.ParameterDict()
        for k, v in demographic_embeddings.items():
            if k not in ['age_bin_idx', 'age_s1']:
                self.demographic_effects[k] = nn.Parameter(torch.zeros(v))


        self.train_ipcw_events = []
        self.train_ipcw_times = []
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters()

    def forward(self, x, y, time, demographics):
        #channel_mask = self.create_channel_mask(x)
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        time = discretize_time(time, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        log_hz_k = self.time_bias
        
        for i, k in enumerate(self.demographic_embeddings):
            if k == 'age_bin_idx':
                #print(demographics.shape, demographics[:, i].shape, demographics[:, i])
                age_emb = self.age_time_interaction(demographics[:, i].to(int)) # demographics[:, i] is [bs]
                log_hz_k = log_hz_k + age_emb
            elif k == 'age_s1':
                age_emb = self.age_time_interaction(demographics[:, i].unsqueeze(1)) # demographics[:, i] is [bs] -> unsqueeze to [bs, 1] for linear layer
                log_hz_k = log_hz_k + age_emb
            else:
                log_hz_k = log_hz_k + self.demographic_effects[k][demographics[:, i].to(int)].unsqueeze(1)
        return log_hz_k
          
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x, y, time, demographic_targets = batch
        log_hz_k = self(x, y, time, demographic_targets)
        return log_hz_k, y, time, demographic_targets

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, time, demographic_targets = batch
        self.train_ipcw_events.append(y.bool().detach().cpu())
        self.train_ipcw_times.append(time.detach().cpu())
        
        if self.transforms is not None:
            batch = self.transforms(batch)
            x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        loss = nll_logistic_hazard(log_hz_k, y, time_bins)
        
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        # use clamped time for loss calculation
        loss = nll_logistic_hazard(log_hz_k, y, time_bins) # [bs, n_classes, pred_len_seconds]

        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        self.events.append(y.bool().detach().cpu())
        self.times.append(time.detach().cpu()) # append original time
        self.preds.append(log_hz_k.detach())
        return loss
        # if torch.any(y):
        #     cindex = self.cindex(log_hz_k, y, time)
        #     self.log("cindex", cindex, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        # else:
        #     self.log("cindex", torch.tensor(0.0, device=x.device), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def on_validation_epoch_end(self):

        preds = torch.cat(self.preds).squeeze()
        preds = torch.sigmoid(preds)
        
        og_device = preds.device.type

        preds = preds.cpu().to(torch.float32).numpy()
        events = torch.cat(self.events).squeeze().numpy()
        times = torch.cat(self.times).squeeze().numpy()
        new_time = self.new_time.cpu().numpy()
        evaluate_risk_years = self.evaluate_risk_years.cpu().numpy()
        
        preds = 1 - hazards_to_survival(preds) # convert hazards to cumulative risk

        try:
            # perform IPCW censoring adjustment/ time clipping
            min_time, max_time = times.min(), times.max()
            if new_time.min() < min_time or new_time.max() > max_time:
                warnings.warn(f"Some evaluation times are outside the range of observed times. Clipping new_time from [{new_time.min()}, {new_time.max()}] to [{min_time}, {max_time}]")
                idx_ = np.where((new_time >= min_time) & (new_time <= max_time))[0]
                preds = preds[:, idx_] # subset preds to these indices
                new_time = new_time[idx_] # subset new_time to these indices
                evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices

            min_first_event_time = times[events == True].min() # get first positive event time (you cant evaluate before this) (but you can after)

            idx_ = np.where((new_time >= min_first_event_time))[0] # find indices of new_time after min_time
            preds = preds[:, idx_] # subset preds to these indices
            new_time = new_time[idx_] # subset new_time to these indices, so evaluations is only after min_time
            evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices
            
            tau = new_time[-1] #times[events == True].max() # time to evaluate c-index at (probability of censoring should be > 0 after this time)
            print(f"Events shape: {events.shape}, sum: {events.sum()}")
            print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
            print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")
        except Exception as e:
            print(f"IPCW or time clipping error: {str(e)}")


        try:
            # create structured arrays for sksurv
            train_events = torch.cat(self.train_ipcw_events).squeeze().numpy()
            train_times = torch.cat(self.train_ipcw_times).squeeze().numpy()

            # clip validation times to 
            
            train_structured = Surv.from_arrays(event=train_events, time=train_times)
            val_structured = Surv.from_arrays(event=events, time=times)
        except Exception as e:
            print(f"Structured array creation error: {str(e)}")
        try:
            # calculate metrics
            auc_all, mean_auc = cumulative_dynamic_auc(train_structured, val_structured, preds, new_time)
            auc_all = torch.tensor(auc_all, device=og_device)
            mean_auc = torch.tensor(mean_auc, device=og_device)
        except Exception as e:
            print(f"AUC calculation error: {str(e)}")
            auc_all = torch.tensor([0.0 for _ in evaluate_risk_years], device=og_device)
            mean_auc = torch.tensor(0.0, device=og_device)
        finally:
            self.log_dict({f"{y}year_auc_ipcw": score for y, score in zip(evaluate_risk_years, auc_all)}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log("mean_ipcw_auc", mean_auc, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        try:
            cindex = concordance_index_ipcw(train_structured, val_structured, preds[:,-1], tau=tau)[0]
            cindex = torch.tensor(cindex, device=og_device)
            self.log("cindex_ipcw", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index calculation error: {str(e)}")
        try:
                # calculate survival probabilities
            preds = 1 - preds # convert cumulative risk to survival
                # calculate brier score
            brier = integrated_brier_score(train_structured, val_structured, preds, new_time)
            brier = torch.tensor(brier, device=og_device)
            self.log("brier_score", brier, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"Brier calculation error: {str(e)}")
        try:
            all_score = mean_auc + (1 - brier) + cindex
            self.log("all_score", all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"All score calculation error: {str(e)}")
        self.preds.clear()
        self.events.clear()
        self.times.clear()

    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
             {
                'params': (self.time_bias)
            },
            {
                'params': (p for n, p in self.demographic_effects.named_parameters()) 
            },
            {
                'params': (p for n, p in self.age_time_interaction.named_parameters())
            }
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class PatchTFTSurvivalMomentum(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of gpus to use
                linear_probing_head, # model head to linear probe/train
                preloaded_model, # loaded pretrained model to use for linear probing
                loss_func,
                evaluate_risk_years = [1,5,10],
                discretize_time=False, # indicator to discretize time for discrete-time survival
                discrete_time_bins=0.25, # function to discretize time for discrete-time survival
                time_range_years=(0,15), # events after this time are censored
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                momentum_steps=4,
                momentum_rate=0.999,
                optimizer_type='AdamW',
                weight_decay=0.001, # weight decay for Adam optimizer
                final_weight_decay=1e-6, # weight decay for final layer
                use_weight_decay_scheduler=False,
                demographic_predictor=False,
                n_demographics=1, 
                demographic_predictor_hidden=64,
                demographic_adversarial_weight=1.0,
                scheduler_kwargs={},
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.momentum_steps = momentum_steps
        self.momentum_rate = momentum_rate
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.discrete_time_bins = discrete_time_bins
        self.time_range_years = time_range_years
        self.discretize_time = discretize_time
        self.demographic_predictor = demographic_predictor
        self.n_demographics = n_demographics
        self.demographic_predictor_hidden = demographic_predictor_hidden
        self.demographic_adversarial_weight = demographic_adversarial_weight
        
        if self.discretize_time:
            if evaluate_risk_years is not None:
                warnings.warn("evaluate_risk_years is ignored when discretize_time=True. Risk will be evaluated at all bins.")
            self.new_time = torch.tensor(make_cuts_days(self.time_range_years[0], self.time_range_years[1], self.discrete_time_bins), dtype=torch.float32)
            self.evaluate_risk_years = self.new_time / 365.25
        else:
            self.new_time = torch.tensor([365.25*y for y in evaluate_risk_years], dtype=torch.float32)
            self.evaluate_risk_years = torch.tensor(evaluate_risk_years, dtype=torch.float32)
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        self.feedforward = Momentum(linear_probing_head, loss=loss_func, batchsize=self.batch_size, steps=self.momentum_steps, rate=self.momentum_rate)    # model head to linear probe/train
        
        if self.demographic_predictor:
            self.demo_adversarial_model = nn.Sequential(
                    nn.Linear(1, self.demographic_predictor_hidden),  # Input: single risk score
                    nn.ReLU(),
                    nn.Dropout(0.1),
                    nn.Linear(self.demographic_predictor_hidden, self.demographic_predictor_hidden // 2),
                    nn.ReLU(),
                    nn.Dropout(0.1),
                    nn.Linear(self.demographic_predictor_hidden // 2, self.n_demographics)  # Output: age, bmi, gender, study
                )
        
        self.train_ipcw_events = []
        self.train_ipcw_times = []
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x, y, time):
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            time = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
        if self.discretize_time:
            time = discretize_time(time, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        loss = self.feedforward(x, y, time) # [bs]
        return loss # Ensure the output is of shape [batch_size]
          
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x = batch[0]
        y = batch[1]
        time = batch[2]
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        log_hz_k = self.feedforward.target(x) # [bs x n labels]
        # if self.discretize_time:
        #     preds = self._discretize_time_convert_logits_to_estimates(log_hz_k, self.new_time, self.cuts_days)
        # else:
        #     preds = torch.sigmoid(log_hz_k)
        return log_hz_k, y, time

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.demographic_predictor:
            x, y, time, demographic_targets = batch
        else:
            x, y, time = batch
        self.train_ipcw_events.append(y.bool().detach().cpu())
        self.train_ipcw_times.append(time.detach().cpu())
        
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
        else:
            time_clamped = time
        
        if self.discretize_time:
            time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        else:
            time_bins = None
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        # use clamped time for loss calculation
        survival_loss = self.feedforward(x, y, time_bins if time_bins is not None else time_clamped) 
        
        if self.demographic_predictor:
            log_hz_k = self.feedforward.target(x) # [bs x n labels]
            cum_risk = 1 - hazards_to_survival_torch(log_hz_k)
            summary_risk = cum_risk[:, -1:]  # Shape: [batch_size, 1] - final year
            demo_predictions = self.demo_adversarial_model(summary_risk)
            demo_loss = demographic_adversarial_loss(demo_predictions, demographic_targets)
            self.log('demo_loss', demo_loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
            loss = survival_loss - self.demographic_adversarial_weight * demo_loss
            if torch.isnan(loss).any():
                warnings.warn(f"NaN values in loss with demographic predictor. targets: {demographic_targets}, predictions: {demo_predictions}")
        else:
            loss = survival_loss
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x = batch[0]
        y = batch[1]
        time = batch[2]
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
        else:
            time_clamped = time
        if self.discretize_time:
            time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        else:
            time_bins = None
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        # use clamped time for loss calculation
        loss = self.feedforward(x, y, time_bins if time_bins is not None else time_clamped) # [bs, n_classes, pred_len_seconds]
        log_hz_k = self.feedforward.target(x)
        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        self.events.append(y.bool().detach().cpu())
        self.times.append(time.detach().cpu()) # append original time
        self.preds.append(log_hz_k.detach())
        # if torch.any(y):
        #     cindex = self.cindex(log_hz_k, y, time)
        #     self.log("cindex", cindex, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        # else:
        #     self.log("cindex", torch.tensor(0.0, device=x.device), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def on_validation_epoch_end(self):

        preds = torch.cat(self.preds).squeeze()
        preds = torch.sigmoid(preds)
        
        og_device = preds.device.type

        preds = preds.cpu().to(torch.float32).numpy()
        events = torch.cat(self.events).squeeze().numpy()
        times = torch.cat(self.times).squeeze().numpy()
        new_time = self.new_time.cpu().numpy()
        evaluate_risk_years = self.evaluate_risk_years.cpu().numpy()

        min_time, max_time = times.min(), times.max()
        if new_time.min() < min_time or new_time.max() > max_time:
            warnings.warn(f"Some evaluation times are outside the range of observed times. Clipping new_time from [{new_time.min()}, {new_time.max()}] to [{min_time}, {max_time}]")
            idx_ = np.where((new_time >= min_time) & (new_time <= max_time))[0]
            preds = preds[:, idx_] # subset preds to these indices
            new_time = new_time[idx_] # subset new_time to these indices
            evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices
        
        if self.discretize_time:
            min_first_event_time = times[events == True].min() # get first positive event time (you cant evaluate before this) (but you can after)
            #max_time = times[events == True].max()

            idx_ = np.where((new_time >= min_first_event_time))[0] # find indices of new_time after min_time
            preds = preds[:, idx_] # subset preds to these indices
            new_time = new_time[idx_] # subset new_time to these indices, so evaluations is only after min_time
            evaluate_risk_years = evaluate_risk_years[idx_] # subset evaluate_risk_years to these indices


        print(f"Events shape: {events.shape}, sum: {events.sum()}")
        print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
        print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")

        preds = 1 - hazards_to_survival(preds) # convert hazards to cumulative risk
        
        try:
            # create structured arrays for sksurv
            train_events = torch.cat(self.train_ipcw_events).squeeze().numpy()
            train_times = torch.cat(self.train_ipcw_times).squeeze().numpy()

            # clip validation times to 
            
            train_structured = Surv.from_arrays(event=train_events, time=train_times)
            val_structured = Surv.from_arrays(event=events, time=times)
        except Exception as e:
            print(f"Structured array creation error: {str(e)}")
        try:
            # calculate metrics
            auc_all, mean_auc = cumulative_dynamic_auc(train_structured, val_structured, preds, new_time)
            auc_all = torch.tensor(auc_all, device=og_device)
            mean_auc = torch.tensor(mean_auc, device=og_device)
        except Exception as e:
            print(f"AUC calculation error: {str(e)}")
            auc_all = torch.tensor([0.0 for _ in evaluate_risk_years], device=og_device)
            mean_auc = torch.tensor(0.0, device=og_device)
        finally:
            self.log_dict({f"{y}year_auc_ipcw": score for y, score in zip(evaluate_risk_years, auc_all)}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log("mean_ipcw_auc", mean_auc, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        try:
            if self.discretize_time:
                # calculate survival probabilities
                preds = 1 - preds # convert cumulative risk to survival
                # calculate brier score
                brier = integrated_brier_score(train_structured, val_structured, preds, new_time)
                self.log("brier_score", torch.tensor(brier, device=og_device), prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"Brier calculation error: {str(e)}")

        self.preds.clear()
        self.events.clear()
        self.times.clear()

    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class HypnogramEncoder(nn.Module):
    def __init__(self, patch_len, d_model, num_categories=5, y_padding_mask=5):
        super().__init__()
        self.y_padding_mask = y_padding_mask
        self.patch_layer = Patch(patch_len=patch_len, stride=patch_len)
        self.embedding = nn.Embedding(num_categories, d_model // 4, padding_idx=y_padding_mask)
        self.encoder = nn.Sequential(
            nn.Linear(d_model // 4, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model),
            nn.GELU()
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Embedding):
            trunc_normal_(m.weight, std=0.02)
        
    def forward(self, x):
        """
        x: [bs x 1 x seq_len]
        returns: [bs, n_channels, d_model, n_patches]
        """
        # replace the padding mask with the number of categories
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.patch_layer(x, constant_pad=True, constant_pad_value=self.y_padding_mask) # [bs x num_patch x n_vars x patch_len]
        embedded = self.embedding(x)  # [bs x num_patch x n_vars x patch_len x embedding_dim]
        # Mean pooling over sequence dimension
        pooled = torch.mean(embedded, dim=-2)  # [bs x num_patch x n_vars x embedding_dim]
        # [bs, n_channels, d_model, n_ffts/n_patches]
        enc = self.encoder(pooled) # [bs x num_patch x 1 x output_dim]
        enc = enc.permute(0, 2, 3, 1) # [bs, n_channels, d_model, n_patches]
        return enc
    
class PatchTFTHypnogramSurvivalMomentum(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus,
                linear_probing_head, # model head to linear probe/train
                preloaded_model, # loaded pretrained model to use for linear probing
                loss_func,
                d_model,
                patch_len,
                hypnogram_index=7,
                hypnogram_only=False,
                signal_only=False,
                zero_hypnogram=False,
                auc_years = [1,5,10],
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                momentum_steps=4,
                momentum_rate=0.999,
                optimizer_type='AdamW',
                weight_decay=0.001, # weight decay for Adam optimizer
                final_weight_decay=1e-6, # weight decay for final layer
                use_weight_decay_scheduler=False,
                create_zero_channel_mask=False, # create a zero channel mask for the encoder when the a data channel is all zeros
                scheduler_kwargs={},
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.hypnogram_index = torch.tensor([hypnogram_index])
        self.x_index = torch.tensor(list(range(self.hypnogram_index)))
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.momentum_steps = momentum_steps
        self.momentum_rate = momentum_rate
        self.create_zero_channel_mask = create_zero_channel_mask
        self.optimizer_type = optimizer_type
        self.auc_years = auc_years
        self.scheduler_kwargs = scheduler_kwargs
        self.hypnogram_only = hypnogram_only
        self.signal_only = signal_only
        self.zero_hypnogram = zero_hypnogram
        assert not (self.signal_only and self.hypnogram_only), "Signal only and hypnogram only cannot be True at the same time"
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        if not self.signal_only:
            self.hypnogram_encoder = HypnogramEncoder(
                patch_len=patch_len,
                num_categories=6,
                d_model=d_model
            )
        else:
            self.hypnogram_encoder = nn.Identity()
        self.loss_func = loss_func
        self.feedforward = Momentum(linear_probing_head, loss=loss_func, batchsize=self.batch_size, steps=self.momentum_steps, rate=self.momentum_rate)    # model head to linear probe/train
        self.cindex = ConcordanceIndex(checks=False)
        self.auroc = Auc()
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        if x.is_nested:
            hyps = []
            x_vals = []
            for x_i in x:
                if self.zero_hypnogram:
                    hyp = torch.zeros_like(x_i.index_select(dim=0, index=self.hypnogram_index.to(x.device)).transpose(0, 1))
                else:
                    hyp = x_i.index_select(dim=0, index=self.hypnogram_index.to(x.device)).transpose(0, 1) 
                hyps.append(hyp)
                x_vals.append(x_i.index_select(dim=0, index=self.x_index.to(x.device)).transpose(0, 1))
            x = torch.nested.as_nested_tensor(x_vals, layout=torch.jagged)
            hypnogram = torch.nested.as_nested_tensor(hyps, layout=torch.jagged)
            x = x.transpose(1, 2)
            hypnogram = hypnogram.transpose(1, 2)
        else:
            if self.zero_hypnogram:
                hypnogram = torch.zeros_like(torch.index_select(x, dim=1, index=self.hypnogram_index.to(x.device)))
            else:
                hypnogram = torch.index_select(x, dim=1, index=self.hypnogram_index.to(x.device))
            x = torch.index_select(x, dim=1, index=self.x_index.to(x.device))
        hypnogram = hypnogram.to(torch.int64)
        #channel_mask = self.create_channel_mask(x)
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        hypnogram_enc = self.hypnogram_encoder(hypnogram)
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        if not self.hypnogram_only and not self.signal_only:
            if x.is_nested:
                x_list = []
                for x_i, h_i in zip(x, hypnogram_enc): # ex torch.Size([6, 1, 256, j14]), the jagged dimension should be 1
                    x_list.append(torch.cat([x_i, h_i], dim=0).permute(2,0,1)) # patch dim (jagged), channels, d_model
                x_cat = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                x_cat = x_cat.permute(0, 2, 3, 1) # bs, channels, d_model, patch dim (jagged)
            else:
                x_cat = torch.cat([x, hypnogram_enc], dim=1) # stack in channel dimension
        elif self.signal_only:
            x_cat = x
        else:
            x_cat = hypnogram_enc
        return x_cat # Ensure the output is of shape [batch_size]
      
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x, y, time = batch
        x_cat = self(x)
        log_hz_k = self.feedforward.target(x_cat) # [bs x n labels]
        return log_hz_k, y, time

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, time = batch
        time[time==0] = 10 # set all 0s to 10 to avoid log(0)
        x_cat = self(x)
        loss = self.feedforward(x_cat, y, time)
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time = batch
        x_cat = self(x)
        loss = self.feedforward(x_cat, y, time) # [bs, n_classes, pred_len_seconds]
        log_hz_k = self.feedforward.target(x_cat)
        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        y = y.bool()
        self.events.append(y)
        self.times.append(time)
        self.preds.append(log_hz_k)
        # if torch.any(y):
        #     cindex = self.cindex(log_hz_k, y, time)
        #     self.log("cindex", cindex, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        # else:
        #     self.log("cindex", torch.tensor(0.0, device=x.device), prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss
    
    def on_validation_epoch_end(self):
        preds = torch.cat(self.preds).squeeze()
        events = torch.cat(self.events).squeeze()
        times = torch.cat(self.times).squeeze()
        og_device = preds.device.type
        print(f"Events shape: {events.shape}, sum: {events.sum()}")
        print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
        print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")
        preds = torch.sigmoid(preds) # add sigmoid activation
        
        preds = preds.to('cpu')
        events = events.to('cpu')
        times = times.to('cpu')
        if preds.dim() == 1:
            preds = preds.unsqueeze(-1)
            events = events.unsqueeze(-1)
            times = times.unsqueeze(-1)
        cindex = torch.tensor([0.0], device=preds.device)
        auc_dict = {}
        for y in self.auc_years:
            auc_dict[f"{y}year_auc"] = torch.tensor([0.0], device=preds.device)
            auc_dict[f"{y}year_auc_ipcw"] = torch.tensor([0.0], device=preds.device)
            auc_dict[f"{y}year_auc_incident"] = torch.tensor([0.0], device=preds.device)
        
        for i in range(preds.shape[-1]): # multilabel
            try:
                cindex += self.cindex(preds[:,i], events[:,i], times[:,i])
            except Exception as e:
                print(f"C-index calculation error: {str(e)}")
                cindex += torch.tensor([0.0], device=preds.device)
            for y in self.auc_years:
                try:
                    new_time = torch.tensor([365.*y], dtype=torch.float32)
                    ipcw = get_ipcw(event=events[:,i], time=times[:,i])
                    ipcw_new_time = get_ipcw(event=events[:,i], time=times[:,i], new_time=new_time)
                    auc_dict[f"{y}year_auc"] += self.auroc(preds[:,i], events[:,i], times[:,i], auc_type="cumulative", new_time=new_time)
                    auc_dict[f"{y}year_auc_ipcw"] += self.auroc(preds[:,i], events[:,i], times[:,i], auc_type="cumulative", new_time=new_time, weight=ipcw, weight_new_time=ipcw_new_time)
                    auc_dict[f"{y}year_auc_incident"] += self.auroc(preds[:,i], events[:,i], times[:,i], auc_type="incident", new_time=new_time)
                except Exception as e:
                    print(f"AUC calculation error: {str(e)}")
                    auc_dict[f"{y}year_auc"] += torch.tensor([0.0], device=preds.device)
                    auc_dict[f"{y}year_auc_ipcw"] += torch.tensor([0.0], device=preds.device)
                    auc_dict[f"{y}year_auc_incident"] += torch.tensor([0.0], device=preds.device)
        try:
            cindex = cindex / preds.shape[1]
            cindex = cindex.to(og_device)
            for y in self.auc_years:
                auc_dict[f"{y}year_auc"] = auc_dict[f"{y}year_auc"] / preds.shape[1]
                auc_dict[f"{y}year_auc_ipcw"] = auc_dict[f"{y}year_auc_ipcw"] / preds.shape[1]
                auc_dict[f"{y}year_auc_incident"] = auc_dict[f"{y}year_auc_incident"] / preds.shape[1]
                auc_dict[f"{y}year_auc"] = auc_dict[f"{y}year_auc"].to(og_device)
                auc_dict[f"{y}year_auc_ipcw"] = auc_dict[f"{y}year_auc_ipcw"].to(og_device)
                auc_dict[f"{y}year_auc_incident"] = auc_dict[f"{y}year_auc_incident"].to(og_device)
            self.log("cindex", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log_dict(auc_dict, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index or auc logging error: {str(e)}")
        finally:
            self.preds.clear()
            self.events.clear()
            self.times.clear()

    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.hypnogram_encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.hypnogram_encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| notest
hyp = torch.randint(0, 5, (5, 101))
#print(hyp.max(), hyp.min(), hyp.shape)

#hyp[2, 10:30] = 5
hypnogram_encoder = HypnogramEncoder(patch_len=5, num_categories=6, d_model=512, y_padding_mask=5)#, y_padding_mask=5)
hypnogram_enc = hypnogram_encoder(hyp)
#hypnogram_enc2 = hypnogram_encoder(hyp)
#print(hypnogram_enc.shape, hypnogram_enc2.shape)
#torch.cat([hypnogram_enc, hypnogram_enc2], dim=1).shape



torch.Size([5, 1, 101])
torch.Size([5, 21, 1, 5])
torch.Size([5, 21, 1, 5, 128])
torch.Size([5, 21, 1, 128])
torch.Size([5, 21, 1, 512])
torch.Size([5, 1, 512, 21])


## Weibull

In [ ]:
#| export
class PatchTFTSurvivalWeibull(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of gpus
                linear_probing_head, # model head to linear probe/train
                preloaded_model, # loaded pretrained model to use for linear probing
                loss_func,
                auc_years = [1,5,10],
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                max_lr=0.01, # maximum learning rate for one_cycle_scheduler
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='Adam',
                weight_decay=0., # weight decay for Adam optimizer
                torch_model_name='model', # name of the pytorch model within the lightning model module, this is to remove layers (for example lightning_model.pytorch_model.head = nn.Identity())
                remove_pretrain_layers=['head'], # layers within the lightning model or lightning model.pytorch_model to remove
                create_zero_channel_mask=False, # create a zero channel mask for the encoder when the a data channel is all zeros
                ):
        super().__init__()
        self.encoder = preloaded_model
        if remove_pretrain_layers is not None and len(remove_pretrain_layers) > 0:
            for l in remove_pretrain_layers:
                if torch_model_name is not None:
                    setattr(getattr(self.encoder, torch_model_name), l, nn.Identity())
                else:
                    setattr(self.encoder, l, nn.Identity())
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.max_lr = max_lr
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size
        self.fine_tune = fine_tune
        self.create_zero_channel_mask = create_zero_channel_mask
        self.optimizer_type = optimizer_type
        self.loss_func = loss_func
        self.auc_years = auc_years
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        self.feedforward = linear_probing_head    # model head to linear probe/train
        self.cindex = ConcordanceIndex(checks=False)
        self.auroc = Auc()
        self.events = []
        self.times = []
        self.cindex_log_hazards = []
        self.auc_log_hazards = {}
        for y in self.auc_years:
            self.auc_log_hazards[f"{y}year_auc"] = []
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        channel_mask = self.create_channel_mask(x)
        x = self.encoder(x, channel_mask=channel_mask) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        log_params = self.feedforward(x) # [bs, n_labels, 2]
        return log_params 
    
    def create_channel_mask(self, x):
        if self.create_zero_channel_mask:
            channel_mask = torch.all(x == 0, dim=-1)
            if not torch.any(channel_mask):
                channel_mask = None
            return channel_mask
        else:
            return None
    
    def _calc_cindex_log_hazards(self, log_params, time):
        n_labels = log_params.size(1)
        l_haz_list = []
        for i in range(n_labels):
            l_haz_list.append(log_hazard(log_params[:,i], time[:,i], all_times=False))
        return torch.stack(l_haz_list, dim = 1) # [bs, n labels]

    def _calc_year_auc_log_hazards(self, log_params, year):
        n_labels = log_params.size(1)
        l_haz_list = []
        for i in range(n_labels):
            l_haz_list.append(log_hazard(log_params[:,i], time=torch.tensor(365.*year, dtype=torch.float32, device=log_params.device), all_times=True))
        return torch.stack(l_haz_list, dim = 1) # [bs, n labels]
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        channel_mask = self.create_channel_mask(x)
        x, _, time = batch
        time[time==0] = 10 
        log_params = self(x) # [bs, n labels, 2]
        log_hz = self._calc_5year_auc_log_hazards(log_params, time)
        return log_hz

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, time = batch
        time[time==0] = 10 # set all 0s to 10 to avoid log(0)
        log_params = self(x) # [bs x n labels x 2]
        loss = self.loss_func(log_params, y, time)
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time = batch
        time[time==0] = 10 # set all 0s to 10 to avoid log(0)
        log_params = self(x) # [bs x n labels x 2]
        loss = self.loss_func(log_params, y, time)
        loss = loss.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        log_hz_cindex = self._calc_cindex_log_hazards(log_params, time)
        log_hz_cindex = log_hz_cindex.to(x.device)
        y = y.bool()
        self.events.append(y)
        self.times.append(time)
        self.cindex_log_hazards.append(log_hz_cindex)

        for y in self.auc_years:
            log_hz_year_auc = self._calc_year_auc_log_hazards(log_params, y)
            log_hz_year_auc = log_hz_year_auc.to(x.device)
            self.auc_log_hazards[f"{y}year_auc"].append(log_hz_year_auc)
        return loss
    
    def on_validation_epoch_end(self):
        year_auc_preds = {}
        for y in self.auc_years:
            year_auc_preds[f"{y}year_auc"] = torch.cat(self.auc_log_hazards[f"{y}year_auc"])
            year_auc_preds[f"{y}year_auc"] = year_auc_preds[f"{y}year_auc"].to('cpu')
        cindex_preds = torch.cat(self.cindex_log_hazards)
        events = torch.cat(self.events)
        times = torch.cat(self.times)
        og_device = cindex_preds.device.type
        cindex_preds = cindex_preds.to('cpu')
        events = events.to('cpu')
        times = times.to('cpu')

        cindex = torch.tensor([0.0], device=cindex_preds.device)
        auc_dict = {}
        for y in self.auc_years:
            auc_dict[f"{y}year_auc"] = torch.tensor([0.0], device=cindex_preds.device)
        for i in range(cindex_preds.shape[1]): # multilabel
            try:
                cindex += self.cindex(cindex_preds[:,i], events[:,i], times[:,i])
            except Exception as e:
                print(f"C-index calculation error: {str(e)}")
                cindex += torch.tensor([0.0], device=cindex_preds.device)
            for y in self.auc_years:
                try:
                    auc_dict[f"{y}year_auc"] += self.auroc(year_auc_preds[f"{y}year_auc"][:,i], events[:,i], times[:,i], auc_type="cumulative", new_time=torch.tensor([365.*y], dtype=torch.float32))
                except Exception as e:
                    print(f"AUC calculation error: {str(e)}")
                    auc_dict[f"{y}year_auc"] += torch.tensor([0.0], device=cindex_preds.device)
        try:
            cindex = cindex / cindex_preds.shape[1]
            for y in self.auc_years:
                auc_dict[f"{y}year_auc"] = auc_dict[f"{y}year_auc"] / cindex_preds.shape[1]
                auc_dict[f"{y}year_auc"] = auc_dict[f"{y}year_auc"].to(og_device)
            cindex = cindex.to(og_device)
            self.log("cindex", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log_dict(auc_dict, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index or auc logging error: {str(e)}")
        finally:
            self.auc_log_hazards.clear()
            self.cindex_log_hazards.clear()
            self.events.clear()
            self.times.clear()

    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(optimizer, max_lr=self.max_lr, total_steps=self.trainer.estimated_stepping_batches)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=self.epochs//10, T_mult=2, eta_min=1e-8)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()